# PCA1 data and how subsequences are built

This notebook shows:
1. **What PCA1 data looks like** — 1D time series in H5 (`LFS`), shape `(1, T_file)` at 100 kHz.
2. **Decimation** — we read every 10th sample → 10 kHz, 7813 samples per window.
3. **How subsequences are tiled** — overlapping windows (nsub, nrecept, stride) in file space.

Run from **soen_fusion_zero** (or set `DECIMATED_ROOT` to your PCA1 H5 directory). If the path does not exist, synthetic data is used so the notebook runs without real files.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Run from soen_fusion_zero so we can import disruptcnn
nb_dir = Path(os.path.abspath(".")).resolve()
if (nb_dir / "disruptcnn").is_dir():
    soen_fusion_zero = nb_dir
else:
    # Try parent (e.g. if notebook is in a subfolder)
    soen_fusion_zero = nb_dir.parent if (nb_dir.parent / "disruptcnn").is_dir() else nb_dir
if str(soen_fusion_zero) not in sys.path:
    sys.path.insert(0, str(soen_fusion_zero))
os.chdir(soen_fusion_zero)
print("Working directory:", soen_fusion_zero)

## 1. Load one PCA1 shot (or synthetic)

PCA1 H5: one file per shot, dataset `LFS` with shape `(1, T_file)` — 1 channel, time at 100 kHz. We decimate by 10× → 7813 samples per training window.

In [ ]:
DECIMATED_ROOT = os.environ.get("DECIMATED_ROOT", "/home/idies/workspace/Storage/yhuang2/persistent/ecei/dsrpt_decimated_pca1")
dec_root = Path(DECIMATED_ROOT)
DATA_STEP = 10
DECIMATE_EXTRA = 10
NSUB_FILE = 78130   # file-space window length (after data_step)
OUTPUT_LEN = NSUB_FILE // DECIMATE_EXTRA  # 7813

if dec_root.exists():
    h5_files = sorted(dec_root.glob("*.h5"))
    if h5_files:
        import h5py
        with h5py.File(h5_files[0], "r") as f:
            lfs = f["LFS"][:]
        print(f"Loaded real PCA1: {h5_files[0].name}")
        print(f"  LFS.shape = {lfs.shape}  (1 channel, T_file at 100 kHz)")
    else:
        lfs = None
        print("No .h5 files in", DECIMATED_ROOT)
else:
    lfs = None
    print("DECIMATED_ROOT not found; using synthetic.")

if lfs is None:
    T_file = 200_000  # synthetic length
    t = np.linspace(0, 10, T_file)
    lfs = np.sin(2 * np.pi * 0.5 * t) + 0.3 * np.random.randn(1, T_file).astype(np.float32)
    lfs = np.asarray(lfs, dtype=np.float64)
    print(f"Synthetic LFS.shape = {lfs.shape}")

In [ ]:
# Single channel 1D signal
signal = np.atleast_2d(lfs)
if signal.shape[0] > 1:
    signal = signal[0:1]
T_file = signal.shape[1]
print(f"Signal shape: (C, T_file) = {signal.shape}")

## 2. Plot raw PCA1 signal (full file)

One channel; time axis in file samples. Decimation: we take every 10th sample to get 7813-point windows for the TCN.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 2.5))
ax.plot(signal[0], color="#2e7d32", alpha=0.9, linewidth=0.5)
ax.set_xlabel("Time (file samples @ 100 kHz)")
ax.set_ylabel("PCA1")
ax.set_title("PCA1 raw signal (1 channel, full file)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. One window: file slice → decimated (7813 points)

Each training sample is a contiguous slice of length `nsub` in **file** space (78130), then we take every 10th → **7813** points. This is what the model sees per sample.

In [ ]:
start_file = 0
stop_file = NSUB_FILE
step = DECIMATE_EXTRA
window_file = signal[0, start_file:stop_file]
window_decimated = window_file[::step]

print(f"File-space window: [{start_file}:{stop_file}] → length {len(window_file)}")
print(f"After [::10]: length {len(window_decimated)} (this is one TCN input)")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 3.5), sharex=False)
ax1.plot(window_file, color="#1565c0", alpha=0.8)
ax1.set_ylabel("PCA1")
ax1.set_title("One window in file space (78130 samples)")
ax1.grid(True, alpha=0.3)
ax2.plot(window_decimated, color="#6a1b9a", alpha=0.9)
ax2.set_xlabel("Time (decimated index)")
ax2.set_ylabel("PCA1")
ax2.set_title("Same window after [::10] → 7813 samples (TCN input)")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. How subsequences are tiled (overlapping windows)

Training uses **overlapping** windows:
- **nsub** = 78130 (file) = window length
- **nrecept** = chosen so stride in file = (nsub - nrecept + 1) matches desired output stride
- Consecutive windows start at `start_i`, `start_i + stride`, …

Below we compute the same tiling as `EceiDatasetOriginal.shots2seqs()` for one shot and draw the first few windows on the full signal.

In [ ]:
# Same as run_tcn_baseline_pca1_decimated_subsample.sh: nrecept from trainer logic
output_recept = 3001  # nrecept_target after decimate scaling
file_stride = ((NSUB_FILE // DECIMATE_EXTRA) - output_recept + 1) * DECIMATE_EXTRA
nrecept_file = max(1, NSUB_FILE - file_stride + 1)
stride_file = NSUB_FILE - nrecept_file + 1

print(f"nsub (file) = {NSUB_FILE}, nrecept (file) = {nrecept_file}")
print(f"Stride (file) = {stride_file}")
print(f"Output length per window = {OUTPUT_LEN}")

# Build window starts for one "shot" of length T_file
N = T_file
num_seq_frac = (N - NSUB_FILE) / float(NSUB_FILE - nrecept_file + 1) + 1
num_seq = max(1, int(np.ceil(num_seq_frac)))
starts = []
for m in range(num_seq):
    start_i = m * (NSUB_FILE - nrecept_file + 1)
    stop_i = start_i + NSUB_FILE
    if stop_i <= T_file:
        starts.append((start_i, stop_i))
print(f"Number of full windows in this file: {len(starts)}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 2.5))
ax.plot(signal[0], color="#2e7d32", alpha=0.5, linewidth=0.4, label="Full signal")

colors = plt.cm.viridis(np.linspace(0.2, 0.8, min(5, len(starts))))
for i, (s, e) in enumerate(starts[:5]):
    ax.axvspan(s, e, alpha=0.25, color=colors[i], label=f"Window {i+1} [{s}:{e}]")
ax.set_xlabel("Time (file samples)")
ax.set_ylabel("PCA1")
ax.set_title("Overlapping subsequence windows (first 5); each window → 7813 decimated points")
ax.legend(loc="upper right", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Example subsequences (decimated) that the TCN sees

Each row is one training sample: 7813 time steps, 1 channel.

In [ ]:
n_show = 3
examples = []
for (s, e) in starts[:n_show]:
    chunk = signal[0, s:e][::DECIMATE_EXTRA]
    examples.append(chunk)

fig, axes = plt.subplots(n_show, 1, figsize=(10, 1.8 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]
for i, (ax, ex) in enumerate(zip(axes, examples)):
    ax.plot(ex, color="#6a1b9a", alpha=0.9)
    ax.set_ylabel(f"Subseq {i+1}")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (7813 steps)")
fig.suptitle("Example subsequences (each shape 7813) — TCN input (1, 7813)", y=1.02)
plt.tight_layout()
plt.show()

## 6. Optional: use EceiDatasetOriginal to get real indices

If you have the shot list and H5 dir, we can show the actual `start_idxi` / `stop_idxi` for one shot from the dataset.

In [ ]:
try:
    from disruptcnn.dataset_original import EceiDatasetOriginal
    disrupt_file = Path(soen_fusion_zero) / "disruptcnn" / "shots" / "d3d_disrupt_ecei.final.txt"
    norm_stats = Path(soen_fusion_zero) / "norm_stats_pca1.npz"
    if not disrupt_file.exists() or not dec_root.exists():
        raise FileNotFoundError("Skip: shot list or decimated root missing")
    nrecept_raw = 3001 * 10
    inner = EceiDatasetOriginal(
        root=os.environ.get("ROOT", "/home/idies/workspace/Storage/yhuang2/persistent/ecei/dsrpt"),
        disrupt_file=str(disrupt_file),
        clear_file=None,
        flattop_only=True,
        normalize=norm_stats.exists(),
        data_step=DATA_STEP,
        nsub=781300,
        nrecept=nrecept_raw,
        decimated_root=str(dec_root),
        clear_decimated_root=None,
        norm_stats_path=str(norm_stats) if norm_stats.exists() else None,
        decimate_extra=DECIMATE_EXTRA,
    )
    # First shot's segments
    shot0_mask = inner.shot_idxi == 0
    starts_ds = inner.start_idxi[shot0_mask]
    stops_ds = inner.stop_idxi[shot0_mask]
    print(f"Shot 0: {inner.shot[0]} → {len(starts_ds)} subsequences")
    print(f"  First 3 windows: start_idxi = {starts_ds[:3].tolist()}, stop_idxi = {stops_ds[:3].tolist()}")
except Exception as e:
    print("Dataset init skipped:", e)